# OneVoice V2 — Vietnamese ASR benchmark
Đo GIPFormer VI-ASR trên split test cố định. Notebook chỉ orchestrate Colab; source, version và report được quản lý rõ ràng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(WORK_ROOT / 'model_cache/huggingface')
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'PyYAML', 'soundfile', 'librosa', 'sherpa-onnx', 'huggingface_hub'], check=True)
MANIFEST = MYDRIVE / 'onevoice_audio_v1/manifest.jsonl'
if not MANIFEST.is_file():
    raise FileNotFoundError('Run colab_data_audit_v2.ipynb first to create/recover manifest.jsonl')
REPORT_ROOT = WORK_ROOT / 'reports/vi_asr'

def run_streaming(command, label):
    print(f'\n[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    print(f'[{label}] exit code: {code}', flush=True)
    if code:
        raise RuntimeError(f'{label} failed; the complete subprocess log is printed above.')

print('Source:', REPO, '| Data:', MANIFEST, '| Reports:', REPORT_ROOT)


In [ ]:
for audio in ('clean', 'noisy'):
    report_dir = REPORT_ROOT / audio
    required = ('aggregate.json', 'predictions.csv', 'run_manifest.json')
    if all((report_dir / name).is_file() for name in required):
        print(f'VI-ASR passthrough/{audio} already complete on Drive; skipping.', flush=True)
        continue
    run_streaming([sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction', 'vi2en', '--split', 'test', '--audio', audio, '--denoiser', 'passthrough', '--report-dir', str(report_dir)], f'VI-ASR passthrough/{audio}')


In [ ]:
import json
{audio: json.loads((REPORT_ROOT / audio / 'aggregate.json').read_text(encoding='utf-8')) for audio in ('clean', 'noisy')}
